# 02 - Data Cleaning

Loads the combined Instagram data saved by `01_data_pull.ipynb`, cleans it (nulls, date parsing), filters it into the 5 trend DataFrames by keyword, then loads and cleans the USDA retail sales data. Saves everything needed for analysis to `cleaned_data/`.

In [1]:
import pandas as pd
import numpy as np


In [13]:
import os
print(os.getcwd())

C:\Users\minni\Downloads\QSS20\covid_food_trends_project\code


## Loading data from previous notebook

In [14]:
df_full = pd.read_csv('../cleaned_data/instagram_combined_raw.csv')
df_full.shape

(298635, 19)

## Data exploration/understanding basic attributes (continued)

In [15]:
# --- Structural sanity check ---
print("Total rows:", len(df_full))
print(df_full.dtypes)

# --- Duplicates ---
#print("Exact duplicate rows:", df_full.duplicated().sum())
#print("Duplicate id:", df_full.duplicated(subset=["id"]).sum())  # 'id' is your best dedup key

# --- Nulls ---
#print(df_full[["text", "creation_time"]].isna().sum())

# --- Empty strings in text ---
#print("Empty text rows:", (df_full["text"].str.strip() == "").sum())

Total rows: 298635
content_type                                str
creation_time                               str
hashtags                                    str
id                                        int64
is_branded_content                         bool
lang                                        str
match_type                                  str
mcl_url                                     str
modified_time                               str
multimedia                                  str
post_owner.id                             int64
post_owner.name                             str
post_owner.type                             str
post_owner.username                         str
statistics.comment_count                float64
statistics.like_count                   float64
statistics.views                        float64
statistics.views_date_last_refreshed    float64
text                                        str
dtype: object


In [16]:
print(type(df_full["hashtags"].iloc[0]))

<class 'str'>


In [17]:
# need to drop rows where text AND hashtags are null
null_text = df_full[df_full["text"].isna()]
print("Null text, has hashtags:", null_text["hashtags"].notna().sum())
print("Null text, hashtags also null:", null_text["hashtags"].isna().sum())

Null text, has hashtags: 96
Null text, hashtags also null: 9200


## Cleaning the Instagram Data

In [18]:
# drop rows with NA values in BOTH of the listed columns (text, hashtags)
df_full = df_full.dropna(subset=["text", "hashtags"], how="all").copy()

# convert creation_time into a date-time object for subsequent processing
df_full["date_parsed"] = pd.to_datetime(df_full["creation_time"], errors="coerce", utc=True)
# create a month column for subsequent processing
df_full["month"] = df_full["date_parsed"].dt.to_period("M")

print("Failed to parse:", df_full["date_parsed"].isna().sum())
print("Date range:", df_full["date_parsed"].min(), "to", df_full["date_parsed"].max()) # date range is Dec 2019 to Dec 2020

Failed to parse: 0
Date range: 2019-12-01 00:00:21+00:00 to 2020-12-01 23:59:34+00:00


C:\Users\minni\AppData\Local\Temp\ipykernel_28844\573679326.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_full["month"] = df_full["date_parsed"].dt.to_period("M")


In [19]:
# Create individual trend dataframes by keyword filtering

# ----------------------------- Setup --------------------------------
trend_keywords = {
    "feta_pasta": ["feta pasta", "baked feta pasta", "baked feta"],
    "sourdough": ["sourdough", "sourdough starter", "sourdough bread"],
    "banana_bread": ["banana bread", "banana bread recipe", "bananabread"],
    "baked_oats": ["baked oats", "baked oatmeal", "bakedoats"],
    "dalgona_coffee": ["dalgona coffee", "whipped coffee", "dalgona"],
}

covid_keywords = ["quarantinebaking", "pandemicbaking", "quarantinecooking"]
covid_pattern = "|".join(covid_keywords)

def filter_by_keywords(df, keywords, text_col="text", hashtag_col="hashtags"):
    pattern = "|".join(keywords)
    text_mask = df[text_col].str.contains(pattern, case=False, na=False, regex=True)
    hashtag_mask = df[hashtag_col].str.contains(pattern, case=False, na=False, regex=True)
    return df[text_mask | hashtag_mask].copy()

# --------------------- Creating the data frames ------------------------------
feta_pasta_df = filter_by_keywords(df_full, trend_keywords["feta_pasta"])
feta_pasta_df["is_covid_framed"] = (
    feta_pasta_df["text"].str.contains(covid_pattern, case=False, na=False, regex=True) |
    feta_pasta_df["hashtags"].str.contains(covid_pattern, case=False, na=False, regex=True)
)
print(f"feta_pasta: {len(feta_pasta_df)} rows total, {feta_pasta_df['is_covid_framed'].sum()} COVID-framed")

sourdough_df = filter_by_keywords(df_full, trend_keywords["sourdough"])
sourdough_df["is_covid_framed"] = (
    sourdough_df["text"].str.contains(covid_pattern, case=False, na=False, regex=True) |
    sourdough_df["hashtags"].str.contains(covid_pattern, case=False, na=False, regex=True)
)
print(f"sourdough: {len(sourdough_df)} rows total, {sourdough_df['is_covid_framed'].sum()} COVID-framed")

banana_bread_df = filter_by_keywords(df_full, trend_keywords["banana_bread"])
banana_bread_df["is_covid_framed"] = (
    banana_bread_df["text"].str.contains(covid_pattern, case=False, na=False, regex=True) |
    banana_bread_df["hashtags"].str.contains(covid_pattern, case=False, na=False, regex=True)
)
print(f"banana_bread: {len(banana_bread_df)} rows total, {banana_bread_df['is_covid_framed'].sum()} COVID-framed")

baked_oats_df = filter_by_keywords(df_full, trend_keywords["baked_oats"])
baked_oats_df["is_covid_framed"] = (
    baked_oats_df["text"].str.contains(covid_pattern, case=False, na=False, regex=True) |
    baked_oats_df["hashtags"].str.contains(covid_pattern, case=False, na=False, regex=True)
)
print(f"baked_oats: {len(baked_oats_df)} rows total, {baked_oats_df['is_covid_framed'].sum()} COVID-framed")

dalgona_coffee_df = filter_by_keywords(df_full, trend_keywords["dalgona_coffee"])
dalgona_coffee_df["is_covid_framed"] = (
    dalgona_coffee_df["text"].str.contains(covid_pattern, case=False, na=False, regex=True) |
    dalgona_coffee_df["hashtags"].str.contains(covid_pattern, case=False, na=False, regex=True)
)
print(f"dalgona_coffee: {len(dalgona_coffee_df)} rows total, {dalgona_coffee_df['is_covid_framed'].sum()} COVID-framed")

feta_pasta: 332 rows total, 5 COVID-framed
sourdough: 136692 rows total, 1587 COVID-framed
banana_bread: 27153 rows total, 1076 COVID-framed
baked_oats: 125 rows total, 39 COVID-framed
dalgona_coffee: 26402 rows total, 200 COVID-framed


In [20]:
print(sourdough_df["month"].value_counts().sort_index())

month
2019-12     6445
2020-01     7372
2020-02     7756
2020-03    10016
2020-04    15525
2020-05    15284
2020-06    11878
2020-07    12390
2020-08    12531
2020-09    12240
2020-10    12674
2020-11    12154
2020-12      427
Freq: M, Name: count, dtype: int64


## Importing and Cleaning Food Sales Data

In [21]:
usda = pd.read_csv('../data/NationalTotalAndSubcategory.csv')

# --- Basic cleaning checks ---
print("Shape:", usda.shape)
print("Nulls per column:\n", usda.isna().sum())
print("Duplicate rows:", usda.duplicated().sum())

# Parse Date column into a real datetime object
usda["Date"] = pd.to_datetime(usda["Date"], errors="coerce")
print("Failed date parses:", usda["Date"].isna().sum())
print("Date range:", usda["Date"].min(), "to", usda["Date"].max())

# Confirm categories/subcategories/variables are as expected
print("Categories:", usda["Category"].unique())
print("Variables:", usda["Variable"].unique())

Shape: (152280, 5)
Nulls per column:
 Date               0
Category           0
Subcategory        0
Variable           0
Value          33696
dtype: int64
Duplicate rows: 0
Failed date parses: 0
Date range: 2019-10-06 00:00:00 to 2023-05-07 00:00:00
Categories: <ArrowStringArray>
[                    'Alcohol',                   'All foods',
                   'Beverages', 'Commercially prepared items',
                       'Dairy',               'Fats and oils',
                      'Fruits',                      'Grains',
       'Meats, eggs, and nuts',                       'Other',
        'Sugar and sweeteners',                  'Vegetables']
Length: 12, dtype: str
Variables: <ArrowStringArray>
[                       'Dollars',              'Dollars last year',
            'Dollars 3 years ago',                     'Unit sales',
           'Unit sales last year',         'Unit sales 3 years ago',
                          'Share',                'Share last year',
           

In [22]:
# Restrict to study window
start_date = "2019-12-01"
end_date = "2021-01-01"

# --- Flour and mixes (sourdough + banana bread proxy) ---
flour = usda[
    (usda["Category"] == "Grains") &
    (usda["Subcategory"] == "Flour and mixes") &
    (usda["Variable"] == "Dollars")
].sort_values("Date")

flour = flour[(flour["Date"] >= start_date) & (flour["Date"] <= end_date)]

# --- Sweet mixes: pancake, muffin, cake (banana bread proxy) ---
sweet_mixes = usda[
    (usda["Category"] == "Commercially prepared items") &
    (usda["Subcategory"] == "Sweet mixes (pancake, muffin, and cake mixes)") &
    (usda["Variable"] == "Dollars")
].sort_values("Date")

sweet_mixes = sweet_mixes[(sweet_mixes["Date"] >= start_date) & (sweet_mixes["Date"] <= end_date)]

# Sanity checks
print("Flour rows:", len(flour), "| range:", flour["Date"].min(), "-", flour["Date"].max())
print("Sweet mixes rows:", len(sweet_mixes), "| range:", sweet_mixes["Date"].min(), "-", sweet_mixes["Date"].max())

print("\nFlour peak week:", flour.loc[flour["Value"].idxmax(), "Date"], 
      "| $", round(flour["Value"].max()/1e6, 1), "million")
print("Sweet mixes peak week:", sweet_mixes.loc[sweet_mixes["Value"].idxmax(), "Date"], 
      "| $", round(sweet_mixes["Value"].max()/1e6, 1), "million")

Flour rows: 57 | range: 2019-12-01 00:00:00 - 2020-12-27 00:00:00
Sweet mixes rows: 57 | range: 2019-12-01 00:00:00 - 2020-12-27 00:00:00

Flour peak week: 2020-03-22 00:00:00 | $ 55.3 million
Sweet mixes peak week: 2020-12-20 00:00:00 | $ 205.5 million


## Saving output for next notebook

Saves the cleaned/filtered Instagram trend DataFrames and the USDA flour/sweet mixes subsets so `03_analysis.ipynb` can load them directly instead of repeating the cleaning steps above.

In [ ]:
feta_pasta_df.to_csv('../cleaned_data/feta_pasta.csv', index=False)
sourdough_df.to_csv('../cleaned_data/sourdough.csv', index=False)
banana_bread_df.to_csv('../cleaned_data/banana_bread.csv', index=False)
baked_oats_df.to_csv('../cleaned_data/baked_oats.csv', index=False)
dalgona_coffee_df.to_csv('../cleaned_data/dalgona_coffee.csv', index=False)

flour.to_csv('../cleaned_data/usda_flour_and_mixes.csv', index=False)
sweet_mixes.to_csv('../cleaned_data/usda_sweet_mixes.csv', index=False)

print("Saved 5 trend CSVs and 2 USDA CSVs to cleaned_data/")